## Notebook for identifying TF communities in the Norman dataset

In [67]:
import pandas as pd
import numpy as np
import igraph as ig
import leidenalg
import math
import networkx as nx
from collections import defaultdict
from scipy.stats import false_discovery_control

In [68]:
#Load and preprocess GRN 
data_folder = '../../../data/real/Norman2019/'
raw_grn = pd.read_csv(data_folder + 'GRN/' + "raw_GRN.csv", index_col=0)

In [69]:
raw_grn

,source,target,coef_mean,coef_abs,p,-logp
0,ZNF467,A2M,0.000060,0.000060,8.808165e-08,7.055115
1,MEF2C,A2M,-0.000067,0.000067,1.970551e-09,8.705412
2,FOXA1,A2M,-0.000032,0.000032,9.588709e-10,9.018240
3,POU3F2,A2M,-0.000055,0.000055,5.251748e-08,7.279696
4,HOXB2,A2M,0.000007,0.000007,4.251820e-01,0.371425
...,...,...,...,...,...,...
131117,SP6,ZYX,0.005368,0.005368,3.642924e-06,5.438550
131118,KLF7,ZYX,0.002558,0.002558,1.330130e-08,7.876106
131119,PRDM1,ZYX,-0.003802,0.003802,1.729851e-07,6.761991
131120,EGR1,ZYX,0.019826,0.019826,8.277994e-13,12.082075


In [70]:
#Drop NA values in P column
raw_grn = raw_grn.dropna(subset=["p"])

In [71]:
raw_grn

,source,target,coef_mean,coef_abs,p,-logp
0,ZNF467,A2M,0.000060,0.000060,8.808165e-08,7.055115
1,MEF2C,A2M,-0.000067,0.000067,1.970551e-09,8.705412
2,FOXA1,A2M,-0.000032,0.000032,9.588709e-10,9.018240
3,POU3F2,A2M,-0.000055,0.000055,5.251748e-08,7.279696
4,HOXB2,A2M,0.000007,0.000007,4.251820e-01,0.371425
...,...,...,...,...,...,...
131117,SP6,ZYX,0.005368,0.005368,3.642924e-06,5.438550
131118,KLF7,ZYX,0.002558,0.002558,1.330130e-08,7.876106
131119,PRDM1,ZYX,-0.003802,0.003802,1.729851e-07,6.761991
131120,EGR1,ZYX,0.019826,0.019826,8.277994e-13,12.082075


In [72]:
#FDR filtering and only leaving P values less than 0.05
adjusted_pvals = false_discovery_control(raw_grn['p'].values, method='bh')

In [73]:
raw_grn.loc[:, 'adjp'] = adjusted_pvals


/tmp/ipykernel_618528/1527661457.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_grn.loc[:, 'adjp'] = adjusted_pvals


In [74]:
raw_grn

,source,target,coef_mean,coef_abs,p,-logp,adjp
0,ZNF467,A2M,0.000060,0.000060,8.808165e-08,7.055115,2.057505e-07
1,MEF2C,A2M,-0.000067,0.000067,1.970551e-09,8.705412,6.458065e-09
2,FOXA1,A2M,-0.000032,0.000032,9.588709e-10,9.018240,3.382435e-09
3,POU3F2,A2M,-0.000055,0.000055,5.251748e-08,7.279696,1.276845e-07
4,HOXB2,A2M,0.000007,0.000007,4.251820e-01,0.371425,4.492387e-01
...,...,...,...,...,...,...,...
131117,SP6,ZYX,0.005368,0.005368,3.642924e-06,5.438550,6.596405e-06
131118,KLF7,ZYX,0.002558,0.002558,1.330130e-08,7.876106,3.644403e-08
131119,PRDM1,ZYX,-0.003802,0.003802,1.729851e-07,6.761991,3.833873e-07
131120,EGR1,ZYX,0.019826,0.019826,8.277994e-13,12.082075,7.595064e-12


In [75]:
filtered_grn = raw_grn[raw_grn['adjp'] <= 0.05]


In [76]:
filtered_grn

,source,target,coef_mean,coef_abs,p,-logp,adjp
0,ZNF467,A2M,0.000060,0.000060,8.808165e-08,7.055115,2.057505e-07
1,MEF2C,A2M,-0.000067,0.000067,1.970551e-09,8.705412,6.458065e-09
2,FOXA1,A2M,-0.000032,0.000032,9.588709e-10,9.018240,3.382435e-09
3,POU3F2,A2M,-0.000055,0.000055,5.251748e-08,7.279696,1.276845e-07
5,IRF1,A2M,0.000078,0.000078,2.393217e-08,7.621018,6.217425e-08
...,...,...,...,...,...,...,...
131116,CHD2,ZYX,0.015406,0.015406,6.155390e-12,11.210744,4.116420e-11
131117,SP6,ZYX,0.005368,0.005368,3.642924e-06,5.438550,6.596405e-06
131118,KLF7,ZYX,0.002558,0.002558,1.330130e-08,7.876106,3.644403e-08
131119,PRDM1,ZYX,-0.003802,0.003802,1.729851e-07,6.761991,3.833873e-07


In [78]:
# Load TF reference
tf_info = pd.read_parquet(data_folder + 'GRN/' + "hg38_TFinfo_dataframe_gimmemotifsv5_fpr2_threshold_10_20210630.parquet") ## Google Drive: gCAL_data > Norman_results

In [41]:
tf_info

,peak_id,gene_short_name,9430076C15RIK,AC002126.6,AC012531.1,AC226150.2,AFP,AHR,AHRR,AIRE,...,ZNF784,ZNF8,ZNF816,ZNF85,ZSCAN10,ZSCAN16,ZSCAN22,ZSCAN26,ZSCAN31,ZSCAN4
0,chr10_100009853_100010953,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10_100081785_100082885,CPN1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,chr10_100185877_100186977,ERLIN1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10_100186978_100187057,ERLIN1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr10_100229510_100230610,CHUK,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39310,chrY_9721196_9722296,TTTY21B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39311,chrY_9735286_9736386,TTTY2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
39312,chrY_9774219_9775319,TTTY1B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39313,chrY_9800153_9801253,TTTY22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
tf_names = set(tf_info.columns.unique())

In [43]:
len(tf_names)

1096

In [44]:
#Only leaving the TFs that are in base GRN file
filtered_tf_grn = filtered_grn.loc[filtered_grn['source'].isin(tf_names) & filtered_grn['target'].isin(tf_names),:]

In [79]:
filtered_tf_grn

,source,target,coef_mean,coef_abs,p,-logp,adjp
0,MLXIP,ENO1,-0.258542,0.258542,9.515030e-11,10.021590,4.381575e-10
1,LYL1,YBX1,0.179087,0.179087,3.808490e-21,20.419247,2.805171e-18
2,HOXB4,LYL1,0.169473,0.169473,1.154312e-20,19.937677,6.735488e-18
3,PITX1,LMO2,0.168925,0.168925,6.008149e-18,17.221259,6.691075e-16
4,NFE2,PITX1,0.144160,0.144160,6.336505e-17,16.198150,3.951411e-15
...,...,...,...,...,...,...,...
4606,ETS1,HIVEP2,0.000003,0.000003,1.562882e-02,1.806074,1.906036e-02
4607,KLF7,OTX2,-0.000003,0.000003,4.674312e-03,2.330282,5.980063e-03
4608,PRDM1,OVOL1,0.000003,0.000003,1.536743e-02,1.813399,1.875634e-02
4609,KLF4,IRX3,-0.000002,0.000002,1.117155e-02,1.951887,1.380846e-02


In [80]:
# Sort by absolute coefficient descending
filtered_tf_grn = filtered_tf_grn.sort_values(by='coef_abs', ascending=False).reset_index(drop=True)

In [81]:
filtered_tf_grn

,source,target,coef_mean,coef_abs,p,-logp,adjp
0,MLXIP,ENO1,-0.258542,0.258542,9.515030e-11,10.021590,4.381575e-10
1,LYL1,YBX1,0.179087,0.179087,3.808490e-21,20.419247,2.805171e-18
2,HOXB4,LYL1,0.169473,0.169473,1.154312e-20,19.937677,6.735488e-18
3,PITX1,LMO2,0.168925,0.168925,6.008149e-18,17.221259,6.691075e-16
4,NFE2,PITX1,0.144160,0.144160,6.336505e-17,16.198150,3.951411e-15
...,...,...,...,...,...,...,...
4606,ETS1,HIVEP2,0.000003,0.000003,1.562882e-02,1.806074,1.906036e-02
4607,KLF7,OTX2,-0.000003,0.000003,4.674312e-03,2.330282,5.980063e-03
4608,PRDM1,OVOL1,0.000003,0.000003,1.536743e-02,1.813399,1.875634e-02
4609,KLF4,IRX3,-0.000002,0.000002,1.117155e-02,1.951887,1.380846e-02


In [84]:
#Create Graph and run Leiden in 10 iterations, each time reducing the last 10% of rows
stepwise_results = []
step_size = math.ceil(len(filtered_tf_grn) * 0.1)

for i in range(10):
    current_df = filtered_tf_grn.iloc[:len(filtered_tf_grn) - i * step_size].copy()
    step_label = f"{100 - i * 10}pct"

    # Build igraph and run Leiden
    edges = list(zip(current_df['source'], current_df['target']))
    weights = current_df['coef_abs'].tolist()
    ig_graph = ig.Graph.TupleList(edges, directed=True)
    ig_graph.es["weight"] = weights

    seed=22
    partition = leidenalg.find_partition(
        ig_graph,
        leidenalg.RBConfigurationVertexPartition,
        weights=weights,
        seed=seed
    )
    
    # Build NetworkX DiGraph for connected components
    
    G_nx = nx.DiGraph()
    for _, row in current_df.iterrows():
        G_nx.add_edge(row['source'], row['target'], weight=row['coef_abs'])
    G_undirected = G_nx.to_undirected()
    connected_components = list(nx.connected_components(G_undirected))

    
    tf_communities = defaultdict(list)
    for comm_id, community in enumerate(partition):
        for v_index in community:
            tf = ig_graph.vs[v_index]["name"]
            if tf in tf_names:
                tf_communities[comm_id].append(tf)

    tf_community_list = []
    for comm_id, tf_list in tf_communities.items():
        for tf in tf_list:
            tf_community_list.append({"community": comm_id, "tf": tf})

    tf_community_df = pd.DataFrame(tf_community_list)
    tf_comm_path = data_folder + 'GRN/' + f"tf_communities_step_{step_label}.csv"
    tf_community_df.to_csv(tf_comm_path, index=False)

    # Finding the edges of a graph which is the TF-TF interaction, assigning names and weight (based on abs_coeff),
    #Lastly checking if TFs are a part of community and if they are of the same community. 
    
    #This is the dictionary created in previous step
    tf_to_community = {tf: comm for comm, tfs in tf_communities.items() for tf in tfs}
    edges_in_community = []

    for edge in ig_graph.es:
        source = ig_graph.vs[edge.source]["name"]
        target = ig_graph.vs[edge.target]["name"]
        weight = edge["weight"]
        if source in tf_to_community and target in tf_to_community:
            if tf_to_community[source] == tf_to_community[target]:
                edges_in_community.append({
                    "community": tf_to_community[source],
                    "source": source,
                    "target": target,
                    "weight": weight
                })

    tf_edge_df = pd.DataFrame(edges_in_community)
    tf_edge_path = data_folder + 'GRN/' + f"tf_community_edges_step_{step_label}.csv"
    tf_edge_df.to_csv(tf_edge_path, index=False)

    # # Plotting, Partition is a community
    # plot_path = f"/home/giorgi_sokhadze/Quantori_ml/data_folder/plot_step_{step_label}.png"
    # layout = ig_graph.layout("fr")
    # ig.plot(
    #     partition,
    #     layout=layout,
    #     target=plot_path,
    #     vertex_size=10,
    #     bbox=(800, 800),
    #     margin=20,
    #     vertex_label=None
    # )

    # Summary saving
    stepwise_results.append({
        "step": step_label,
        "n_edges": len(current_df),
        "n_leiden_communities": len(partition),
        "tf_counts_per_community": {k: len(v) for k, v in tf_communities.items()},
        "tf_communities_file": tf_comm_path,
        "tf_community_edges_file": tf_edge_path,
        "n_connected_components": len(connected_components),
        #"plot": plot_path
    })



In [85]:
# Final DataFrame
summary_df = pd.DataFrame(stepwise_results)
summary_df.to_csv(data_folder + 'GRN/' + "tf_tf_grn_summary.csv", index=False)

In [ ]:
# get 10% of rows, chosen on the basis of tf_tf_grn_summary.csv


step_size = math.ceil(len(filtered_tf_grn) * 0.1)

i = 9

filtered_tf_grn_10 = filtered_tf_grn.iloc[:len(filtered_tf_grn) - i * step_size].copy()

filtered_tf_grn_10

,source,target,coef_mean,coef_abs,p,-logp,adjp
0,MLXIP,ENO1,-0.258542,0.258542,9.515030e-11,10.021590,4.381575e-10
1,LYL1,YBX1,0.179087,0.179087,3.808490e-21,20.419247,2.805171e-18
2,HOXB4,LYL1,0.169473,0.169473,1.154312e-20,19.937677,6.735488e-18
3,PITX1,LMO2,0.168925,0.168925,6.008149e-18,17.221259,6.691075e-16
4,NFE2,PITX1,0.144160,0.144160,6.336505e-17,16.198150,3.951411e-15
...,...,...,...,...,...,...,...
448,NFE2,MEF2C,0.012127,0.012127,1.524765e-03,2.816797,2.042872e-03
449,ZEB1,GATA2,0.012058,0.012058,4.631155e-13,12.334311,4.666339e-12
450,ID1,HES4,-0.012049,0.012049,7.080045e-07,6.149964,1.422690e-06
451,HES1,ATF6B,0.011976,0.011976,2.061734e-07,6.685767,4.510247e-07


In [87]:
# involved TFs
sources = filtered_tf_grn_10['source'].unique()
targets = filtered_tf_grn_10['target'].unique()
tfs = list(set(sources).union(targets))
len(tfs)

89

In [88]:
# how many genes are regulated by these 88 tfs?
regulated_genes = filtered_grn['target'][filtered_grn['source'].isin(tfs)].unique()
len(regulated_genes)

2701

In [89]:
# how many genes in total?
all_genes = filtered_grn['target'].unique()
len(all_genes)

2701

In [90]:
# all tfs in unique genes?
len(set(tfs).union(set(all_genes)))

2701

In [91]:
file_path = data_folder + 'GRN/' + "all_genes_norman.txt"
with open(file_path, "w") as file:
    for item in all_genes:
        file.write(item + "\n")
